# AEF-BNG: Databricks Pipeline

This notebook runs the AEF-BNG pipeline on a Databricks cluster. It reprojects
AEF satellite embedding tiles from UTM to the British National Grid at 10m
resolution and writes the results to a Unity Catalog Delta table.

### How it works

1. The AEF tile index (a STAC GeoParquet file on Source Cooperative S3) is queried
   directly with predicate pushdown — only tiles overlapping the requested BNG
   bounds and years are returned. No local download is needed.
2. The BNG extent is divided into 10km grid squares. Each square becomes a Spark
   task that reads the overlapping COG tiles from S3, reprojects them to BNG,
   merges any overlap at UTM zone boundaries (first-valid strategy), and extracts
   one row per valid 10m cell.
3. Processing uses `mapInArrow` for Arrow-native throughput — no Pandas serde.
   The tile index is broadcast once and deserialised once per partition.
4. Results are written to a Unity Catalog Delta table with liquid clustering on
   `(year, bng_ref)` for efficient spatial and temporal queries.

### Requirements

- Databricks Runtime 17.3+ (Spark 4.0, Python 3.12)
- Cluster with outbound access to `us-west-2.opendata.source.coop` (public S3, no credentials needed)
- Unity Catalog permissions to create/write to the target table

## 1. Install the package

Install `aef-bng` on the cluster. On a shared cluster you may prefer to attach
it as a cluster library instead.

In [ ]:
# %pip install /Workspace/Repos/<your-user>/aef-bng
# dbutils.library.restartPython()

## 2. Configure the pipeline

Set the bounds, years, and Unity Catalog table name.

The bounds below cover Greater London — adjust for your area of interest.
BNG bounds are in EPSG:27700 metres: `(minx, miny, maxx, maxy)`.

In [ ]:
CATALOG = "my_catalog"
SCHEMA = "my_schema"
TABLE = "aef_bng"

YEARS = [2025]
BOUNDS = (521722, 171089, 540290, 187123)  # Greater London

TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"
print(f"Target table: {TABLE_NAME}")
print(f"Years:        {YEARS}")
print(f"Bounds:       {BOUNDS}")

In [ ]:
from aef_bng.config import AEFBNGConfig

config = AEFBNGConfig(
    years=YEARS,
    bounds=BOUNDS,
    table_name=TABLE_NAME,
)
config

## 3. Preview the processing grid

Check how many 10km chunks will be processed. Each chunk produces up to
1,000,000 rows (1000 x 1000 pixels at 10m). Chunks over sea or outside AEF
coverage are skipped automatically.

In [ ]:
from aef_bng.grid import BNGOutputGrid

grid = BNGOutputGrid(config.bounds, config.chunk_size)
chunks = grid.enumerate_chunks()

num_tasks = len(chunks) * len(config.years)
print(f"10km chunks:      {len(chunks)}")
print(f"Years:            {len(config.years)}")
print(f"Total tasks:      {num_tasks}")
print(f"Max possible rows: {num_tasks * 1_000_000:,}")
print()
print("Sample chunks:")
for c in chunks[:10]:
    print(f"  {c.bng_10km_ref}  bounds={c.bounds_bng}")

## 4. Preview the tile index

Query the AEF STAC GeoParquet index to see which tiles overlap the requested
bounds. This is the same query the pipeline will run internally — it goes
directly to S3 with predicate pushdown, so only matching rows are downloaded.

In [ ]:
from aef_bng.index import AEFBNGIndex

index = AEFBNGIndex()
tiles_gdf = index.load_for_bounds(config.bounds, config.years)

print(f"Tiles matching bounds + years: {len(tiles_gdf)}")
print(f"Columns: {list(tiles_gdf.columns)}")
tiles_gdf.head()

## 5. Ensure the Unity Catalog schema exists

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")  # type: ignore  # noqa: F821, PGH003

## 6. Run the pipeline

This distributes chunk processing across the cluster using `mapInArrow`.

On the **driver**:
- The tile index is queried from S3 (predicate pushdown), pickled, and broadcast.
- A DataFrame of `(chunk, year)` combinations is created and repartitioned.

On each **executor**:
- The index is deserialised once per partition (amortised across all chunks).
- For each chunk: overlapping COGs are read from S3, reprojected to BNG,
  merged (first-valid), and extracted as Arrow batches.

The result is written to the Unity Catalog table with liquid clustering on
`(year, bng_ref)`.

Monitor progress in the Spark UI — look for the `mapInArrow` stage.

In [ ]:
from aef_bng.spark import process_with_spark

process_with_spark(config)

## 7. Verify the output

In [ ]:
df = spark.table(TABLE_NAME)  # type: ignore  # noqa: F821, PGH003

print("Schema:")
df.printSchema()
print(f"Total rows: {df.count():,}")

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

# BNG references should be exactly 10 characters
df.select(
    F.min(F.length("bng_ref")).alias("min_ref_len"),
    F.max(F.length("bng_ref")).alias("max_ref_len"),
    F.min(F.size("embedding")).alias("min_emb_len"),
    F.max(F.size("embedding")).alias("max_emb_len"),
    F.count("*").alias("total_rows"),
).show()

In [ ]:
# Year distribution
df.groupBy("year").count().orderBy("year").show()

In [ ]:
# Confirm liquid clustering is active
detail = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}")  # type: ignore  # noqa: F821, PGH003
detail.select("clusteringColumns").show(truncate=False)

## 8. Query examples

Liquid clustering on `(year, bng_ref)` means these queries skip irrelevant
files automatically.

In [ ]:
from pyspark.sql import functions as F

spark.table(TABLE_NAME).filter(F.col("bng_ref") == "TQ30008000").show(truncate=False)  # type: ignore  # noqa: F821, PGH003

## 9. Processing all of Great Britain

To process the full BNG extent, use the default bounds. This covers
~9,100 10km chunks. At 5 years that is ~45,500 Spark tasks — a cluster
with 32+ cores will process this efficiently.

```python
from aef_bng.constants import BNG_BOUNDS

config = AEFBNGConfig(
    years=[2020, 2021, 2022, 2023, 2024],
    bounds=BNG_BOUNDS,  # (0, 0, 700_000, 1_300_000)
    table_name="my_catalog.my_schema.aef_bng",
)
process_with_spark(config)
```